# Intégrer le modèle spécialisé dans l'agent : serveur, robustesse, architecture mixte

Le fil rouge de la formation : l'agent du service client de Biclou & Cie, qui lit le message d'un client, appelle les outils du magasin (la fiche d'une commande, le stock, les règles du magasin, un ticket, le transfert à un conseiller), puis répond selon la charte. Le chapitre 2 l'a construit, ses outils servis par MCP et sa boucle ReAct ; le chapitre 3 a constitué son jeu d'appels d'outils ; le chapitre 4 a affiné son modèle, Qwen3-1.7B, par QLoRA. Ce TP branche ce modèle spécialisé dans l'agent : derrière un serveur d'inférence compatible avec l'API d'OpenAI, à la place du modèle de base, sans réécrire la logique de l'agent. Puis il rend l'agent robuste à ses erreurs et à celles de ses outils, et le combine avec un modèle généraliste qui planifie.

1. Servir le modèle spécialisé
2. Servir comme à l'entraînement
3. La bascule
4. Robustesse
5. Architecture mixte

Ce notebook se fait de préférence sur GPU : *Exécution > Modifier le type d'exécution > GPU T4*. llama.cpp y sert Qwen3-1.7B, le modèle du cours, avec ses programmes compilés pour CUDA, en une demi-heure environ (une estimation) ; vLLM, le serveur des GPU en production, s'installerait en remplaçant le `torch` de Colab (les diapositives le montrent). Sans GPU, il tourne quand même, en plus petit : llama.cpp sert Qwen3-0.6B sur le processeur (`MODELE`, dans la cellule de configuration), en 40 minutes environ sur les deux processeurs de Colab (une estimation) ; les exercices sont les mêmes. Chaque exercice mesure quelques demandes seulement : de quoi voir la méthode et l'ordre de grandeur des coûts ; la comparaison des modèles sur tout le jeu figé est l'objet du chapitre suivant.

## Données, modèles et outils

Les données de l'agent, dans le dossier `agent/` du jeu `bike-shop-support` : `eval.jsonl`, le jeu d'évaluation figé (139 demandes avec leurs appels attendus et leurs résultats), sur lequel ce TP mesure et rien n'est réglé ; `demandes.jsonl`, l'historique, d'où l'exercice 5 tire ses demandes composées ; `commandes.csv`, `stock.csv` et `charte.md`, ce que lisent les outils. `agent-sft-validation.jsonl`, le jeu de validation du chapitre 4 : la consigne de l'agent et les schémas des outils, tels que le modèle les a vus à l'entraînement. `qwen3_training.jinja` : le gabarit de Qwen3 avec lequel `trl` 1.18 a entraîné l'adaptateur au chapitre 4.

Et llama.cpp, à l'étiquette `b12359`, sa version 1.0.0 (13 novembre 2026) : `llama.cpp/`, son dépôt, pour ses scripts de conversion au format GGUF ; `llama-b12359/`, ses programmes compilés pour Linux (`llama-server`) : pour CUDA sur GPU, pour le processeur sans GPU.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/bike-shop-support/agent/eval.jsonl
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/bike-shop-support/agent/demandes.jsonl
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/bike-shop-support/agent/commandes.csv
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/bike-shop-support/agent/stock.csv
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/bike-shop-support/charte.md
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/bike-shop-support/lab-results/agent-sft-validation.jsonl
!wget -q https://raw.githubusercontent.com/huggingface/trl/v1.18.0/trl/chat_templates/qwen3_training.jinja
!git clone -q --depth 1 --branch b12359 https://github.com/ggml-org/llama.cpp
import torch

GPU = torch.cuda.is_available()
# Les programmes de llama.cpp : pour CUDA 12.8 sur GPU, pour le processeur sans GPU.
if GPU:
  !wget -q -O llama.tar.gz https://github.com/ggml-org/llama.cpp/releases/download/b12359/llama-b12359-bin-ubuntu-cuda-12.8-x64.tar.gz
else:
  !wget -q -O llama.tar.gz https://github.com/ggml-org/llama.cpp/releases/download/b12359/llama-b12359-bin-ubuntu-x64.tar.gz
!tar -xzf llama.tar.gz
%pip install -q "mcp>=2,<3" openai sentencepiece ./llama.cpp/gguf-py

In [ ]:
import json
import os
import re
import subprocess
import sys
import time
import urllib.request
from dataclasses import dataclass
from difflib import unified_diff

import jsonschema
import openai
import pandas as pd
import requests
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
from mcp import Client, StdioServerParameters
from openai import OpenAI
from openai.types.chat import ChatCompletion
from pydantic import BaseModel, ConfigDict, ValidationError
from transformers import AutoTokenizer

# Le modèle de l'agent : sur GPU, celui du chapitre 4 ; sans GPU, Qwen3-0.6B, environ
# trois fois plus rapide sur le processeur, avec l'adaptateur de référence de ce modèle.
MODELE = "Qwen/Qwen3-1.7B" if GPU else "Qwen/Qwen3-0.6B"
TAILLE = MODELE.split("-")[-1].lower()  # "1.7b" ou "0.6b"
# L'étiquette de llama.cpp clonée et téléchargée ci-dessus, et ses programmes.
LLAMA_CPP = "b12359"
BINAIRES = f"llama-{LLAMA_CPP}"
# Les processeurs dont dispose le notebook, 2 sur Colab : llama.cpp, lui, compterait
# les cœurs physiques de la machine.
CPUS = len(os.sched_getaffinity(0))
# Sur GPU, llama-server y place toutes les couches du modèle.
OPTIONS_GPU = ["--n-gpu-layers", "99"] if GPU else []
print(MODELE, "sur GPU" if GPU else f"sans GPU, sur {CPUS} processeurs")


def lance(commande: list[str]) -> str:
  """Lance une commande et renvoie sa sortie ; si elle échoue, lève une erreur."""
  resultat = subprocess.run(commande, capture_output=True, text=True)
  sortie = resultat.stdout + resultat.stderr
  if resultat.returncode != 0:
    raise RuntimeError(f"{commande[1]} a échoué :\n{sortie[-3000:]}")
  return sortie


def taille_mo(chemin: str) -> float:
  return os.path.getsize(chemin) / 1e6


def secret(nom: str) -> str | None:
  """Un secret de Colab (la clé dans la barre de gauche), sinon une variable
  d'environnement ; None s'il n'existe pas."""
  try:
    from google.colab import userdata
  except ImportError:
    return os.environ.get(nom)
  try:
    return userdata.get(nom)
  except Exception:  # pas de tel secret, ou pas d'accès
    return None

L'adaptateur, dans le dossier `ADAPTATEUR` : le vôtre, si vous l'avez publié à la fin du chapitre 4 (le secret Colab `HF_TOKEN`, l'étiquette `v1.0`) et qu'il a été entraîné sur `MODELE`, sinon celui de référence, de [shuuchuu/models](https://github.com/shuuchuu/models), entraîné par le même code. Un adaptateur LoRA ne s'applique qu'au modèle de base sur lequel il a été appris : le vôtre, de Qwen3-1.7B si le chapitre 4 a tourné sur GPU, ne sert pas sans GPU, où ce notebook sert Qwen3-0.6B. La cellule dit lequel elle prend, et pourquoi.

In [ ]:
ADAPTATEUR = "agent-sft-lora"
REFERENCE = f"https://github.com/shuuchuu/models/raw/refs/heads/main/agent-sft-lora-qwen3-{TAILLE}"


def adaptateur_du_hub() -> str | None:
  """Télécharge dans ADAPTATEUR la version v1.0 de votre adaptateur, s'il existe
  pour MODELE ; sinon, dit pourquoi et rend None."""
  jeton = secret("HF_TOKEN")
  if not jeton:
    print("Pas de secret HF_TOKEN : pas de dépôt à lire sur le Hub.")
    return None
  api = HfApi(token=jeton)
  depot = f"{api.whoami()['name']}/agent-sft-lora"
  if not api.repo_exists(depot):
    print(f"Pas de dépôt {depot} sur le Hub.")
    return None
  if "v1.0" not in [etiquette.name for etiquette in api.list_repo_refs(depot).tags]:
    print(f"{depot} n'a pas d'étiquette v1.0.")
    return None
  with open(api.hf_hub_download(depot, "adapter_config.json", revision="v1.0")) as f:
    base = json.load(f)["base_model_name_or_path"]
  if base != MODELE:  # un adaptateur ne s'applique qu'à son modèle de base
    print(f"{depot} adapte {base}, ce notebook sert {MODELE}.")
    return None
  api.snapshot_download(depot, revision="v1.0", local_dir=ADAPTATEUR)
  return depot


depot = adaptateur_du_hub()
if depot is None:
  os.makedirs(ADAPTATEUR, exist_ok=True)
  for fichier in ["adapter_config.json", "adapter_model.safetensors"]:
    urllib.request.urlretrieve(f"{REFERENCE}/{fichier}", f"{ADAPTATEUR}/{fichier}")
print("Adaptateur :", depot or f"celui de référence, {REFERENCE}")

llama.cpp lit le format GGUF : un seul fichier, qui porte les poids, le tokenizer et le *chat template* du modèle. La cellule ci-dessous convertit le modèle de base en GGUF, quantifié en Q8_0 (8 bits par poids et un facteur d'échelle par bloc de 32 poids : moitié moins lourd qu'en 16 bits, des sorties presque identiques), et l'adaptateur en un GGUF à part, en 16 bits, avec les scripts de llama.cpp. L'adaptateur n'est pas fusionné dans les poids : le serveur l'applique à chaque calcul. Le fichier reste petit, un même modèle de base sert avec ou sans lui, et un nouvel adaptateur se livre sans reconvertir la base ; la fusion irait un peu plus vite à l'inférence.

In [ ]:
os.makedirs("gguf", exist_ok=True)
GGUF_BASE = f"gguf/qwen3-{TAILLE}-q8_0.gguf"
GGUF_LORA = "gguf/agent-sft-lora-f16.gguf"
# Le modèle de base du Hub, sa configuration et son tokenizer compris.
DOSSIER_MODELE = snapshot_download(
  MODELE, allow_patterns=["*.json", "*.safetensors", "*.txt"]
)
debut = time.time()
lance(
  [
    sys.executable,
    "llama.cpp/convert_hf_to_gguf.py",
    DOSSIER_MODELE,
    "--outfile",
    GGUF_BASE,
    "--outtype",
    "q8_0",
  ]
)
lance(
  [
    sys.executable,
    "llama.cpp/convert_lora_to_gguf.py",
    ADAPTATEUR,
    "--base",
    DOSSIER_MODELE,
    "--outfile",
    GGUF_LORA,
    "--outtype",
    "f16",
  ]
)
print(f"Conversions en {time.time() - debut:.0f} s")
print(f"Modèle de base, Q8_0 : {taille_mo(GGUF_BASE):.0f} Mo")
print(f"Adaptateur, F16 : {taille_mo(GGUF_LORA):.0f} Mo")

Le système du magasin et son serveur MCP, repris tels quels du chapitre 2 : `biclou.py`, ses données et les fonctions des outils ; `serveur_biclou.py`, le serveur MCP qui les déclare, en stdio.

In [ ]:
%%writefile biclou.py
"""Le système de Biclou & Cie : ses données, et ce que font les outils de l'agent."""

import re
import unicodedata

import pandas as pd

commandes = pd.read_csv("commandes.csv", dtype=str, keep_default_na=False)
commandes = commandes.set_index("numero")
stock = pd.read_csv("stock.csv", keep_default_na=False)
with open("charte.md", encoding="utf-8") as f:
  charte = f.read()

CATEGORIES = [
  "information",
  "livraison",
  "reparation",
  "retour",
  "location",
  "facturation",
  "compte",
]
PRIORITES = ["basse", "normale", "haute"]
RAISONS = ["securite", "colis_perdu", "paiement", "accident_location", "vol_location"]
MODELES = list(dict.fromkeys(stock["modele"]))
TAILLES = list(dict.fromkeys(stock["taille"]))
MAGASINS = list(dict.fromkeys(stock["magasin"]))
RUBRIQUES = {
  "Retours": "retours",
  "Livraison": "livraison",
  "Garantie": "garantie",
  "Atelier": "atelier",
  "Location": "location",
  "Facturation": "facturation",
  "Compte": "compte",
  "Magasins": "magasins",
  "Tailles de cadre": "tailles",
}


def lit_politique(charte: str) -> dict[str, list[str]]:
  """Les phrases de chaque rubrique de la partie « Politique » de la charte."""
  texte = charte.replace("\xa0", " ").split("## Politique", 1)[1]
  puces = re.findall(r"^- \*\*(.+?)\*\* : (.+?)(?=^- |\Z)", texte, re.M | re.S)
  return {
    RUBRIQUES[titre]: re.split(
      r"(?<=\.) (?=[A-ZÀ-Ý])", f"{titre} : {' '.join(corps.split())}"
    )
    for titre, corps in puces
  }


PASSAGES = lit_politique(charte)
THEMES = list(PASSAGES)


def racines(texte: str) -> set[str]:
  """Les mots de 4 lettres ou plus, comparés par leurs 5 premières lettres."""
  texte = unicodedata.normalize("NFKD", texte.lower())
  texte = "".join(c for c in texte if not unicodedata.combining(c))
  return {mot[:5] for mot in re.findall(r"[a-z]+", texte) if len(mot) >= 4}


def chercher_commande(numero: str) -> dict:
  """La fiche de la commande, sans ses champs vides, ou une erreur."""
  if not re.fullmatch(r"CMD-\d{6}", numero):
    return {"erreur": "numéro invalide : CMD- suivi de 6 chiffres"}
  if numero not in commandes.index:
    return {"erreur": f"commande introuvable : {numero}"}
  fiche = {"numero": numero, **commandes.loc[numero].to_dict()}
  fiche = {cle: valeur for cle, valeur in fiche.items() if valeur != ""}
  fiche["montant"] = float(fiche["montant"])
  fiche["paiements"] = int(fiche["paiements"])
  return fiche


def verifier_stock(modele: str, taille: str, magasin: str) -> dict:
  """La quantité en stock et, en rupture, la date du prochain réassort."""
  ligne = stock[
    (stock["modele"] == modele)
    & (stock["taille"] == taille)
    & (stock["magasin"] == magasin)
  ]
  if ligne.empty:
    return {"erreur": "modèle, taille ou magasin inconnu"}
  quantite = int(ligne["quantite"].iloc[0])
  resultat = dict(modele=modele, taille=taille, magasin=magasin, quantite=quantite)
  if quantite == 0:
    resultat["reassort"] = ligne["reassort"].iloc[0]
  return resultat


def chercher_politique(theme: str, question: str) -> dict:
  """Les trois passages de la rubrique les plus proches de la question."""
  if theme not in PASSAGES:
    return {"erreur": f"rubrique inconnue : {theme}"}
  mots = racines(question)
  tries = sorted(
    enumerate(PASSAGES[theme]), key=lambda p: (-len(mots & racines(p[1])), p[0])
  )
  return {"theme": theme, "passages": [passage for _, passage in tries[:3]]}


compteurs = {"ticket": 30000, "dossier": 5000}
EQUIPES = {
  "securite": ("atelier", "aujourd'hui"),
  "colis_perdu": ("livraison", "sous 24 h"),
  "paiement": ("comptabilite", "sous 24 h"),
  "accident_location": ("location", "sous 24 h"),
  "vol_location": ("location", "sous 24 h"),
}


def creer_ticket(categorie: str, priorite: str, resume: str) -> dict:
  """Le numéro du ticket créé."""
  if categorie not in CATEGORIES or priorite not in PRIORITES:
    return {"erreur": "catégorie ou priorité inconnue"}
  compteurs["ticket"] += 1
  return {"ticket": f"T-{compteurs['ticket']}", "statut": "ouvert"}


def escalader(raison: str, resume: str) -> dict:
  """Le numéro du dossier transmis et l'équipe qui le prend en charge."""
  if raison not in EQUIPES:
    return {"erreur": f"raison inconnue : {raison}"}
  compteurs["dossier"] += 1
  equipe, recontact = EQUIPES[raison]
  return {
    "dossier": f"ESC-{compteurs['dossier']}",
    "equipe": equipe,
    "recontact": recontact,
  }

In [ ]:
%%writefile serveur_biclou.py
"""Le serveur MCP des outils de Biclou & Cie, en stdio."""

from typing import Annotated, Literal

import biclou
from mcp.server.mcpserver import MCPServer
from pydantic import Field

serveur = MCPServer("biclou", log_level="WARNING")


@serveur.tool()
def chercher_commande(
  numero: Annotated[
    str, Field(description="le numéro de commande, CMD- suivi de 6 chiffres")
  ],
) -> dict:
  """Cherche une commande dans le système de commandes."""
  return biclou.chercher_commande(numero)


@serveur.tool()
def verifier_stock(
  modele: Annotated[
    Literal[tuple(biclou.MODELES)], Field(description="le modèle de vélo")
  ],
  taille: Annotated[
    Literal[tuple(biclou.TAILLES)], Field(description="la taille de cadre")
  ],
  magasin: Annotated[
    Literal[tuple(biclou.MAGASINS)], Field(description="la ville du magasin")
  ],
) -> dict:
  """Vérifie le stock d'un modèle de vélo dans un magasin."""
  return biclou.verifier_stock(modele, taille, magasin)


@serveur.tool()
def chercher_politique(
  theme: Annotated[
    Literal[tuple(biclou.THEMES)],
    Field(description="la rubrique des règles où chercher"),
  ],
  question: Annotated[str, Field(description="la question, en quelques mots")],
) -> dict:
  """Cherche dans les règles du magasin les passages qui répondent à une question."""
  return biclou.chercher_politique(theme, question)


@serveur.tool()
def creer_ticket(
  categorie: Annotated[
    Literal[tuple(biclou.CATEGORIES)],
    Field(description="la catégorie de la demande"),
  ],
  priorite: Annotated[
    Literal[tuple(biclou.PRIORITES)], Field(description="la priorité de la demande")
  ],
  resume: Annotated[str, Field(description="la demande en une phrase")],
) -> dict:
  """Crée un ticket pour une demande que l'équipe doit traiter."""
  return biclou.creer_ticket(categorie, priorite, resume)


@serveur.tool()
def escalader(
  raison: Annotated[
    Literal[tuple(biclou.RAISONS)], Field(description="la raison du transfert")
  ],
  resume: Annotated[str, Field(description="la situation en une phrase")],
) -> dict:
  """Transmet la demande à un conseiller, pour un problème de sécurité ou d'argent."""
  return biclou.escalader(raison, resume)


if __name__ == "__main__":
  serveur.run()

Le client MCP du chapitre 2 : `SERVEUR`, les paramètres de lancement du serveur ; `vers_outil`, un outil MCP au format des schémas d'outils de l'API ; `execute(mcp, appel)`, le résultat d'un appel `{"name": ..., "arguments": {...}}` par le serveur, ou `{"erreur": ...}`. Et ce dont la boucle de l'agent a besoin : la consigne de l'agent, `CONSIGNES_AGENT`, lue dans le jeu de validation, `conversation(message)`, `nettoie`, la réponse de repli `REPLI` et le budget `MAX_ETAPES`. `SANS_PENSEE` éteint la pensée de Qwen3, comme à l'entraînement : un serveur passe `chat_template_kwargs` au *chat template*.

In [ ]:
eval_fige = pd.read_json("eval.jsonl", lines=True)
demandes = pd.read_json("demandes.jsonl", lines=True)
nombre_appels = eval_fige["appels_attendus"].str.len()
with open("agent-sft-validation.jsonl", encoding="utf-8") as f:
  validation = [json.loads(ligne) for ligne in f]
# La consigne et les schémas des outils, tels que le modèle les a vus à l'entraînement.
CONSIGNES_AGENT = validation[0]["messages"][0]["content"]
OUTILS_ENTRAINEMENT = json.loads(validation[0]["tools"])

SERVEUR = StdioServerParameters(command=sys.executable, args=["serveur_biclou.py"])


def sans_titres(schema: dict) -> dict:
  proprietes = {
    nom: {cle: v for cle, v in p.items() if cle != "title"}
    for nom, p in schema["properties"].items()
  }
  sans_titre = {cle: v for cle, v in schema.items() if cle != "title"}
  return {**sans_titre, "properties": proprietes}


def vers_outil(outil) -> dict:
  """Un outil MCP au format des schémas d'outils."""
  fonction = dict(
    name=outil.name,
    description=outil.description,
    parameters=sans_titres(outil.input_schema),
  )
  return dict(type="function", function=fonction)


async def execute(mcp, appel: dict) -> dict:
  """Le résultat d'un appel par le serveur, ou l'erreur que l'agent recevra."""
  resultat = await mcp.call_tool(appel["name"], appel["arguments"])
  texte = resultat.content[0].text
  return {"erreur": texte} if resultat.is_error else json.loads(texte)


async with Client(SERVEUR) as mcp:
  outils_mcp = (await mcp.list_tools()).tools
OUTILS_MCP = [vers_outil(o) for o in outils_mcp]
SCHEMAS = {o.name: o.input_schema for o in outils_mcp}


def conversation(message: str, systeme: str = CONSIGNES_AGENT) -> list[dict]:
  return [dict(role="system", content=systeme), dict(role="user", content=message)]


def nettoie(texte: str) -> str:
  """Sans les espaces de fin de ligne que le modèle laisse parfois."""
  return "\n".join(ligne.rstrip() for ligne in texte.strip().splitlines())


SANS_PENSEE = {"chat_template_kwargs": {"enable_thinking": False}}
# Générations au plus par message du client : deux appels, une relance, la réponse.
MAX_ETAPES = 4
REPLI = """Bonjour,

Merci pour votre message. Nous n'avons pas pu traiter votre demande automatiquement : un conseiller de notre équipe la reprend dès maintenant et vous répondra par écrit dans les meilleurs délais, avec les informations de votre dossier. Vous n'avez rien à faire d'ici là.

Bonne journée,
L'équipe Biclou & Cie"""
print(len(eval_fige), "demandes dans le jeu figé ; les outils :", list(SCHEMAS))

## 1. Servir le modèle spécialisé

`llama-server`, le serveur de llama.cpp, sert un fichier GGUF par une API HTTP compatible avec celle d'OpenAI : une application qui appelle déjà un modèle avec le client `openai` n'a qu'à changer d'adresse. Il applique le *chat template* du modèle et lit les appels d'outils dans sa sortie : les blocs `<tool_call>` de Qwen3 arrivent dans le champ `tool_calls` de la réponse, et l'agent n'analyse plus aucun texte. Avec `--lora`, il applique l'adaptateur au modèle de base.

Écrivez `demarre_serveur(nom, port, options, gguf)`, qui lance `llama-server` en arrière-plan sur le fichier `gguf` (par défaut `GGUF_BASE`), sur le port `port`, sous l'alias `nom` (le nom du modèle dans l'API), avec `CPUS` threads, un seul *slot* (`--parallel 1`), un contexte de 4 096 tokens, un cache des prompts en mémoire vive borné à 1 024 Mio (`--cache-ram 1024`) et les options du GPU (`OPTIONS_GPU`, vide sans GPU), plus les `options` données ; son journal va dans le fichier `<nom>.log`. Elle arrête d'abord le serveur du même nom s'il tourne (`arrete`), range le processus dans `SERVEURS`, attend que le point `/health` réponde avec le code 200, et renvoie le processus. Un seul *slot* garde en cache le début commun des prompts, la consigne et les schémas des outils, environ 1 300 tokens, que le serveur ne calcule qu'une fois. Le cache de `--cache-ram` (8 Gio par défaut) garde, en mémoire vive, les prompts qu'une autre requête a remplacés dans le *slot*, pour les reprendre plus tard : borné, il laisse tenir plusieurs serveurs dans la mémoire de Colab.

Démarrez deux serveurs : `biclou-agent` sur le port 8081, avec l'adaptateur (`--lora`, `GGUF_LORA`), et `qwen3-base` sur le port 8080, sans. Vérifiez ce qu'ils servent : leur point `/v1/models`, et `/lora-adapters` pour l'adaptateur. Créez pour chacun un `Moteur` (ci-dessous), `SPECIALISE` et `BASE` : le client `openai` de son adresse et le nom de son modèle. Envoyez enfin le message d'`essai` deux fois au moteur spécialisé, puis une fois au moteur de base, avec la consigne de l'agent (`conversation`) et les outils de son serveur MCP (`OUTILS_MCP`), sans échantillonnage, 200 tokens au plus, la pensée éteinte (`SANS_PENSEE`) ; affichez chaque fois la durée, les tokens du prompt et ceux qui venaient du cache (`usage`), le texte du message rendu et ses appels.

Lequel appelle un outil ? Sous quelle forme l'appel arrive-t-il ? Pourquoi la deuxième requête va-t-elle bien plus vite que la première ?

**Documentation utile**

- [`llama-server`](https://github.com/ggml-org/llama.cpp/blob/b12359/tools/server/README.md)
- [`subprocess.Popen`](https://docs.python.org/3/library/subprocess.html#subprocess.Popen)

In [ ]:
@dataclass
class Moteur:
  """Un modèle servi : le client de son serveur et le nom du modèle."""

  client: OpenAI
  modele: str


SERVEURS = {}  # les processus llama-server, par nom


def arrete(nom: str) -> None:
  """Arrête le serveur `nom` s'il tourne, et attend qu'il ait fini."""
  if nom in SERVEURS:
    serveur = SERVEURS.pop(nom)
    serveur.terminate()
    serveur.wait()


essai = eval_fige[nombre_appels == 1].iloc[0]
print(essai["message"], "\n\nAppel attendu :", essai["appels_attendus"])

In [ ]:
# Votre code ici

### Conseils supplémentaires

- La commande : `[f"{BINAIRES}/llama-server", "--model", gguf, "--port", str(port), "--alias", nom, "--threads", str(CPUS), "--parallel", "1", "--ctx-size", "4096", "--cache-ram", "1024", *OPTIONS_GPU, *options]`, lancée par `subprocess.Popen(commande, stdout=journal, stderr=subprocess.STDOUT)`, avec `journal = open(f"{nom}.log", "w")`
- `requests.get(f"http://localhost:{port}/health")` lève `requests.ConnectionError` tant que le serveur n'écoute pas, puis répond 503 pendant qu'il charge le modèle : réessayez chaque seconde ; `serveur.poll()` vaut `None` tant que le processus vit, sinon le serveur s'est arrêté, et son journal dit pourquoi
- `requests.get(f"http://localhost:{port}/v1/models").json()["data"]` : les modèles servis ; `/lora-adapters` : la liste des adaptateurs chargés, avec leur `scale`
- Un client : `OpenAI(base_url=f"http://localhost:{port}/v1", api_key="aucune")` ; le serveur ne demande pas de clé, mais le client en veut une
- `moteur.client.chat.completions.create(model=moteur.modele, messages=..., tools=OUTILS_MCP, temperature=0, max_tokens=200, extra_body=SANS_PENSEE)` : `extra_body` ajoute à la requête des champs que l'API d'OpenAI n'a pas
- La réponse : `.choices[0].message`, son texte `.content` et ses appels `.tool_calls` (`None` sans appel), chacun avec `.function.name` et `.function.arguments`, du texte JSON ; `.usage.prompt_tokens`, et `.usage.prompt_tokens_details.cached_tokens`, ceux que le serveur a repris de son cache
- Sans GPU, sur deux processeurs, la première requête à un serveur prend une minute environ avec Qwen3-0.6B ; sur GPU, quelques secondes

```python
def demarre_serveur(
  nom: str, port: int, options: list[str] = (), gguf: str = GGUF_BASE
) -> subprocess.Popen:
  arrete(nom)
  journal = open(f"{nom}.log", "w")
  commande = [f"{BINAIRES}/llama-server", "--model", gguf, ...]
  serveur = subprocess.Popen(commande, stdout=journal, stderr=subprocess.STDOUT)
  SERVEURS[nom] = serveur
  while True:
    try:
      if requests.get(f"http://localhost:{port}/health").status_code == 200:
        return serveur
    except requests.ConnectionError:
      pass
    if serveur.poll() is not None:
      raise RuntimeError(f"llama-server s'est arrêté : lisez {nom}.log")
    time.sleep(1)


demarre_serveur("biclou-agent", 8081, ["--lora", GGUF_LORA])
demarre_serveur("qwen3-base", 8080)
for port in [8081, 8080]:
  print(requests.get(f"http://localhost:{port}/v1/models").json()["data"][0]["id"])
print(requests.get(...).json())

SPECIALISE = Moteur(
  OpenAI(base_url="http://localhost:8081/v1", api_key="aucune"), "biclou-agent"
)
BASE = Moteur(...)
for moteur in [SPECIALISE, SPECIALISE, BASE]:
  debut = time.time()
  reponse = moteur.client.chat.completions.create(...)
  message = reponse.choices[0].message
  usage = reponse.usage
  print(
    f"\n{moteur.modele} : {time.time() - debut:.0f} s, {usage.prompt_tokens} tokens"
  )
  print("dont en cache :", usage.prompt_tokens_details.cached_tokens)
  print("Texte :", repr(message.content))
  appels = [(a.function.name, a.function.arguments) for a in message.tool_calls or []]
  print("Appels :", appels)
```

### Solution

In [ ]:
def demarre_serveur(
  nom: str, port: int, options: list[str] = (), gguf: str = GGUF_BASE
) -> subprocess.Popen:
  """Lance llama-server sur `gguf`, sous l'alias `nom`, et attend qu'il soit prêt."""
  arrete(nom)
  journal = open(f"{nom}.log", "w")
  commande = [
    f"{BINAIRES}/llama-server",
    "--model",
    gguf,
    "--port",
    str(port),
    "--alias",
    nom,
    "--threads",
    str(CPUS),
    "--parallel",
    "1",
    "--ctx-size",
    "4096",
    "--cache-ram",
    "1024",
    *OPTIONS_GPU,
    *options,
  ]
  serveur = subprocess.Popen(commande, stdout=journal, stderr=subprocess.STDOUT)
  SERVEURS[nom] = serveur
  while True:
    try:
      if requests.get(f"http://localhost:{port}/health").status_code == 200:
        return serveur
    except requests.ConnectionError:
      pass
    if serveur.poll() is not None:
      raise RuntimeError(f"llama-server s'est arrêté : lisez {nom}.log")
    time.sleep(1)


demarre_serveur("biclou-agent", 8081, ["--lora", GGUF_LORA])
demarre_serveur("qwen3-base", 8080)
for port in [8081, 8080]:
  print(requests.get(f"http://localhost:{port}/v1/models").json()["data"][0]["id"])
print(requests.get("http://localhost:8081/lora-adapters").json())

SPECIALISE = Moteur(
  OpenAI(base_url="http://localhost:8081/v1", api_key="aucune"), "biclou-agent"
)
BASE = Moteur(
  OpenAI(base_url="http://localhost:8080/v1", api_key="aucune"), "qwen3-base"
)
for moteur in [SPECIALISE, SPECIALISE, BASE]:
  debut = time.time()
  reponse = moteur.client.chat.completions.create(
    model=moteur.modele,
    messages=conversation(essai["message"]),
    tools=OUTILS_MCP,
    temperature=0,
    max_tokens=200,
    extra_body=SANS_PENSEE,
  )
  message = reponse.choices[0].message
  usage = reponse.usage
  print(
    f"\n{moteur.modele} : {time.time() - debut:.0f} s, {usage.prompt_tokens} tokens"
  )
  print("dont en cache :", usage.prompt_tokens_details.cached_tokens)
  print("Texte :", repr(message.content))
  appels = [(a.function.name, a.function.arguments) for a in message.tool_calls or []]
  print("Appels :", appels)

# Le serveur a lu le bloc <tool_call> de la sortie : l'appel arrive dans tool_calls,
# ses arguments en texte JSON, et le texte du message est vide. La première requête
# calcule tout le prompt ; la deuxième reprend du cache du slot le début qu'elle
# partage avec la première (ici, tout le prompt). Les deux serveurs lisent le même
# fichier de base : le système ne le charge qu'une fois en mémoire.

## 2. Servir comme à l'entraînement

L'adaptateur a appris une suite de tokens précise : celle que `trl` a construite au chapitre 4, avec la consigne, les schémas des outils de l'entraînement et son propre gabarit de Qwen3, `qwen3_training.jinja`. À l'inférence, le serveur construit le prompt avec le gabarit embarqué dans le fichier GGUF, celui du modèle, et avec les outils que l'agent lui envoie. Est-ce la même suite ?

### 2.1 Le prompt du serveur et celui de l'entraînement

`historique` (ci-dessous) : un exemple du jeu de validation du chapitre 4 à deux appels, jusqu'au résultat du second ; le modèle doit alors répondre au client. Construisez :

- `prompt_entrainement`, le texte que `trl` a appris pour cet historique : le *chat template* du tokenizer de `MODELE` appliqué avec le gabarit d'entraînement (`GABARIT_ENTRAINEMENT`), les outils de l'entraînement (`OUTILS_ENTRAINEMENT`), l'invite de génération et la pensée éteinte ;
- `prompt_serveur(port, outils)`, le prompt que construit le serveur du port `port` pour cet historique et ces outils, la pensée éteinte : son point `/apply-template` le rend sans rien générer ;
- `ecarts(prompt)`, qui affiche ce qui diffère entre `prompt` et `prompt_entrainement` (`unified_diff`), ligne à ligne, après avoir coupé les deux textes après chaque virgule suivie d'une espace : une ligne de schéma d'outil devient une ligne par clé.

Affichez les écarts du serveur spécialisé avec les outils de son serveur MCP (`OUTILS_MCP`), puis avec ceux de l'entraînement. Quels écarts viennent du gabarit, lesquels des outils ? Les outils du serveur MCP et ceux de l'entraînement décrivent-ils les mêmes arguments (comparez-les comme des dictionnaires, sans la clé `return` de ceux de l'entraînement, `sans_retour`) ?

**Documentation utile**

- [`apply_chat_template`](https://huggingface.co/docs/transformers/main_classes/tokenizer#transformers.TokenizersBackend.apply_chat_template)
- [`/apply-template`](https://github.com/ggml-org/llama.cpp/blob/b12359/tools/server/README.md#post-apply-template-apply-chat-template-to-a-conversation)
- [`difflib.unified_diff`](https://docs.python.org/3/library/difflib.html#difflib.unified_diff)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(DOSSIER_MODELE)
with open("qwen3_training.jinja", encoding="utf-8") as f:
  GABARIT_ENTRAINEMENT = f.read()
exemple = next(
  e for e in validation if sum(m["role"] == "tool" for m in e["messages"]) == 2
)
historique = exemple["messages"][:-1]
print([m["role"] for m in historique])


def sans_retour(outil: dict) -> dict:
  """Un schéma d'outil sans la clé `return`, que l'API d'OpenAI n'a pas."""
  fonction = {cle: v for cle, v in outil["function"].items() if cle != "return"}
  return dict(type=outil["type"], function=fonction)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `tokenizer.apply_chat_template(historique, tools=OUTILS_ENTRAINEMENT, chat_template=GABARIT_ENTRAINEMENT, tokenize=False, add_generation_prompt=True, enable_thinking=False)` : le prompt, en texte
- `requests.post(f"http://localhost:{port}/apply-template", json={"messages": historique, "tools": outils, **SANS_PENSEE}).json()["prompt"]` : le prompt du serveur
- `texte.replace(", ", ",\n").splitlines()` : les lignes, coupées après chaque virgule
- `unified_diff(a, b, "entrainement", "serveur", lineterm="", n=0)` : les lignes qui diffèrent entre les listes `a` et `b`, `-` pour celles de `a`, `+` pour celles de `b`, sans contexte
- `OUTILS_MCP == [sans_retour(o) for o in OUTILS_ENTRAINEMENT]` : deux dictionnaires sont égaux quand ils ont les mêmes clés et les mêmes valeurs, dans n'importe quel ordre

```python
prompt_entrainement = tokenizer.apply_chat_template(
  historique,
  tools=OUTILS_ENTRAINEMENT,
  chat_template=...,
  tokenize=False,
  add_generation_prompt=True,
  enable_thinking=False,
)


def prompt_serveur(port: int, outils: list[dict]) -> str:
  corps = {"messages": historique, "tools": outils, **SANS_PENSEE}
  return requests.post(...).json()["prompt"]


def coupe(texte: str) -> list[str]:
  return texte.replace(", ", ",\n").splitlines()


def ecarts(prompt: str) -> None:
  lignes = unified_diff(
    coupe(prompt_entrainement),
    coupe(prompt),
    "entrainement",
    "serveur",
    lineterm="",
    n=0,
  )
  for ligne in lignes:
    print(ligne)


for nom, outils in [("OUTILS_MCP", OUTILS_MCP), ...]:
  print(f"\n=== Le serveur spécialisé, avec {nom}")
  ecarts(prompt_serveur(8081, outils))
print("\nMêmes arguments :", OUTILS_MCP == ...)
```

#### Solution

In [ ]:
prompt_entrainement = tokenizer.apply_chat_template(
  historique,
  tools=OUTILS_ENTRAINEMENT,
  chat_template=GABARIT_ENTRAINEMENT,
  tokenize=False,
  add_generation_prompt=True,
  enable_thinking=False,
)


def prompt_serveur(port: int, outils: list[dict]) -> str:
  """Le prompt que le serveur construit pour `historique`, sans rien générer."""
  corps = {"messages": historique, "tools": outils, **SANS_PENSEE}
  reponse = requests.post(f"http://localhost:{port}/apply-template", json=corps)
  return reponse.json()["prompt"]


def coupe(texte: str) -> list[str]:
  """Les lignes du texte, coupées après chaque virgule."""
  return texte.replace(", ", ",\n").splitlines()


def ecarts(prompt: str) -> None:
  """Ce qui diffère entre `prompt` et prompt_entrainement."""
  lignes = unified_diff(
    coupe(prompt_entrainement),
    coupe(prompt),
    "entrainement",
    "serveur",
    lineterm="",
    n=0,
  )
  for ligne in lignes:
    print(ligne)


for nom, outils in [
  ("OUTILS_MCP", OUTILS_MCP),
  ("OUTILS_ENTRAINEMENT", OUTILS_ENTRAINEMENT),
]:
  print(f"\n=== Le serveur spécialisé, avec {nom}")
  ecarts(prompt_serveur(8081, outils))
print(
  "\nMêmes arguments :", OUTILS_MCP == [sans_retour(o) for o in OUTILS_ENTRAINEMENT]
)

# Deux écarts. Le gabarit : celui du modèle ôte le <think> vide des appels passés, que
# celui de trl garde ; tout historique d'au moins un appel en est changé. Les outils :
# le serveur MCP range les clés de ses schémas dans un autre ordre (la description
# avant le type), et le serveur d'inférence ne transmet pas la clé return, que l'API
# d'OpenAI n'a pas. Les mêmes arguments, pas le même texte.

### 2.2 Servir le gabarit de l'entraînement

Le gabarit se corrige au serveur : `--chat-template-file` remplace celui du fichier GGUF. Les outils, côté agent : il enverra désormais ceux de l'entraînement (`OUTILS`, ci-dessous), qui décrivent les mêmes arguments que ceux du serveur MCP, dans l'ordre de l'entraînement. Reste la clé `return` : aucun serveur compatible avec l'API d'OpenAI ne la transmet, il faudrait la retirer du jeu d'entraînement et réentraîner.

Mesurez l'effet du gabarit sur l'étape qu'il touche, le deuxième appel d'une demande. Écrivez `deuxiemes_appels(moteur)`, qui, pour chaque demande de `deux_appels` (ci-dessous : des demandes du jeu figé à deux appels), envoie au moteur l'historique correct jusqu'au résultat du premier appel attendu (`jusqua_second`) et rend le premier appel de sa réponse (`lit_appel`), `None` sans appel : 96 tokens au plus, sans échantillonnage, la pensée éteinte, les outils `OUTILS`. Puis :

1. rangez dans `resultats`, sous la clé `"gabarit du modèle"`, les deuxièmes appels du moteur spécialisé tel qu'il tourne ;
2. redémarrez le serveur spécialisé avec l'adaptateur et le gabarit de l'entraînement (`--chat-template-file`, le fichier `qwen3_training.jinja`), et vérifiez que son prompt pour `historique` est maintenant celui de l'entraînement, aux outils près (le prompt d'entraînement construit avec `OUTILS`) ;
3. rangez ses deuxièmes appels sous la clé `"gabarit de l'entraînement"`.

Comptez, pour chaque gabarit, les deuxièmes appels corrects (`meme_appel`). Que peut dire une mesure sur si peu de demandes ? Quel gabarit servir, et pourquoi ?

In [ ]:
OUTILS = [sans_retour(o) for o in OUTILS_ENTRAINEMENT]  # ce que l'agent enverra
N_DEUX = 3  # demandes à deux appels mesurées
deux_appels = eval_fige[nombre_appels == 2].head(N_DEUX)
ARGUMENTS_LIBRES = {"question", "resume"}  # du texte, qu'on ne compare pas mot à mot


def exacts(appel: dict) -> dict:
  """Les arguments vérifiables d'un appel : sans les arguments libres."""
  arguments = appel["arguments"] if isinstance(appel["arguments"], dict) else {}
  return {a: v for a, v in arguments.items() if a not in ARGUMENTS_LIBRES}


def meme_appel(obtenu: dict | None, attendu: dict) -> bool:
  """Le même outil, avec les mêmes arguments vérifiables."""
  return (
    obtenu is not None
    and obtenu["name"] == attendu["name"]
    and exacts(obtenu) == exacts(attendu)
  )


def lit_appel(appel_outil) -> dict:
  """Un appel de `tool_calls` en dictionnaire ; ses arguments restent du texte quand
  ils ne sont pas un JSON valide."""
  try:
    arguments = json.loads(appel_outil.function.arguments)
  except json.JSONDecodeError:
    arguments = appel_outil.function.arguments
  return dict(name=appel_outil.function.name, arguments=arguments)


def appel_chat(appel: dict) -> dict:
  """Un appel au format des conversations : ses arguments en texte JSON."""
  arguments = json.dumps(appel["arguments"], ensure_ascii=False)
  return dict(type="function", function=dict(name=appel["name"], arguments=arguments))


def jusqua_second(demande: dict) -> list[dict]:
  """La conversation correcte jusqu'au résultat du premier appel attendu."""
  appel, resultat = demande["appels_attendus"][0], demande["resultats_outils"][0]
  return conversation(demande["message"]) + [
    dict(role="assistant", content="", tool_calls=[appel_chat(appel)]),
    dict(role="tool", content=json.dumps(resultat, ensure_ascii=False)),
  ]


for demande in deux_appels.to_dict("records"):
  print([a["name"] for a in demande["appels_attendus"]], demande["message"][:70])

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- Une requête : `moteur.client.chat.completions.create(model=moteur.modele, messages=jusqua_second(demande), tools=OUTILS, temperature=0, max_tokens=96, extra_body=SANS_PENSEE)`
- `demandes.to_dict("records")` : les demandes d'un `DataFrame`, en dictionnaires
- Le serveur redémarré : `demarre_serveur("biclou-agent", 8081, ["--lora", GGUF_LORA, "--chat-template-file", "qwen3_training.jinja"])` ; le client de `SPECIALISE` reste valable, l'adresse n'a pas changé
- Le prompt attendu : `tokenizer.apply_chat_template(historique, tools=OUTILS, chat_template=GABARIT_ENTRAINEMENT, tokenize=False, add_generation_prompt=True, enable_thinking=False)`, à comparer à `prompt_serveur(8081, OUTILS)`
- `meme_appel(obtenu, demande["appels_attendus"][1])` : le deuxième appel est-il l'attendu ?

```python
def deuxiemes_appels(moteur: Moteur) -> list[dict | None]:
  appels = []
  for demande in deux_appels.to_dict("records"):
    reponse = moteur.client.chat.completions.create(...)
    appels_outils = reponse.choices[0].message.tool_calls or []
    appels.append(lit_appel(appels_outils[0]) if appels_outils else None)
  return appels


resultats = {"gabarit du modèle": deuxiemes_appels(SPECIALISE)}
demarre_serveur("biclou-agent", 8081, [...])
attendu = tokenizer.apply_chat_template(...)
print("Le prompt de l'entraînement :", prompt_serveur(8081, OUTILS) == attendu)
resultats["gabarit de l'entraînement"] = deuxiemes_appels(SPECIALISE)
for gabarit, appels in resultats.items():
  attendus = [d["appels_attendus"][1] for d in deux_appels.to_dict("records")]
  corrects = sum(meme_appel(o, a) for o, a in zip(appels, attendus))
  print(f"\n{gabarit} : {corrects} deuxièmes appels corrects sur {len(appels)}")
  for appel in appels:
    print("  ", appel)
```

#### Solution

In [ ]:
def deuxiemes_appels(moteur: Moteur) -> list[dict | None]:
  """Le deuxième appel du moteur pour chaque demande de deux_appels, après le premier
  appel attendu et son résultat."""
  appels = []
  for demande in deux_appels.to_dict("records"):
    reponse = moteur.client.chat.completions.create(
      model=moteur.modele,
      messages=jusqua_second(demande),
      tools=OUTILS,
      temperature=0,
      max_tokens=96,
      extra_body=SANS_PENSEE,
    )
    appels_outils = reponse.choices[0].message.tool_calls or []
    appels.append(lit_appel(appels_outils[0]) if appels_outils else None)
  return appels


debut = time.time()
resultats = {"gabarit du modèle": deuxiemes_appels(SPECIALISE)}
demarre_serveur(
  "biclou-agent",
  8081,
  ["--lora", GGUF_LORA, "--chat-template-file", "qwen3_training.jinja"],
)
attendu = tokenizer.apply_chat_template(
  historique,
  tools=OUTILS,
  chat_template=GABARIT_ENTRAINEMENT,
  tokenize=False,
  add_generation_prompt=True,
  enable_thinking=False,
)
print("Le prompt de l'entraînement :", prompt_serveur(8081, OUTILS) == attendu)
resultats["gabarit de l'entraînement"] = deuxiemes_appels(SPECIALISE)
for gabarit, appels in resultats.items():
  attendus = [d["appels_attendus"][1] for d in deux_appels.to_dict("records")]
  corrects = sum(meme_appel(o, a) for o, a in zip(appels, attendus))
  print(f"\n{gabarit} : {corrects} deuxièmes appels corrects sur {len(appels)}")
  for appel in appels:
    print("  ", appel)
print(f"\n{time.time() - debut:.0f} s")

# Sur trois demandes, un appel de plus ou de moins ne prouve rien : la mesure se fait
# sur toutes les demandes à plusieurs appels du jeu figé (chapitre suivant). Le choix,
# lui, ne dépend pas d'elle : on sert le prompt sur lequel le modèle a été entraîné.
# L'autre solution est d'entraîner avec le gabarit du modèle. Le serveur du modèle de
# base garde le gabarit du modèle : c'est celui que ce modèle a appris.

## 3. La bascule

L'agent du chapitre 2 générait avec `transformers` et lisait lui-même les blocs `<tool_call>`. Derrière un serveur compatible avec l'API d'OpenAI, sa boucle ne dépend plus du modèle : écrivez `agent(moteur, mcp, messages)`, la boucle ReAct du chapitre 2 par l'API, une fonction `async` qui répète au plus `MAX_ETAPES` fois :

1. demander la suite de `messages` au moteur (`moteur.client`, `moteur.modele`), avec les outils `OUTILS`, sans échantillonnage, `max_tokens` tokens au plus, la pensée éteinte, et ajouter ses tokens au trajet (`compte_tokens`) ;
2. sans appel dans la réponse, son texte (`nettoie`) est la réponse au client, ou `REPLI` s'il est vide, et la boucle s'arrête ;
3. sinon, ajouter à `messages` le message rendu, tel quel (`model_dump(exclude_none=True)`) ; lire ses appels (`lit_appel`) : des arguments illisibles arrêtent la boucle sur `REPLI`, comme au chapitre 2 (`format_lisible` faux) ; sinon, exécuter chaque appel par le serveur MCP (`execute`) et ajouter son résultat, un message `tool` en texte JSON, avec l'identifiant de l'appel (`tool_call_id`).

Le budget épuisé, la réponse est `REPLI`. La réponse, quelle qu'elle soit, est ajoutée à `messages`. La fonction rend le trajet : ses `appels`, `format_lisible`, ses `generations`, sa `reponse` et ses tokens. Avec `bavard=True`, elle affiche chaque appel, son résultat et la réponse.

Puis basculez : faites tourner l'agent sur `eval_part` (ci-dessous : 6 demandes du jeu figé, 2 sans appel, 2 à un appel, 2 à deux appels, celles que le chapitre 4 a mesurées sans GPU), avec `BASE`, puis avec `SPECIALISE`, et rangez dans `bascule` une ligne par demande et par moteur (`mesure`, avec en plus la conversation, `messages`), puis la moyenne par moteur. Qu'a-t-il fallu changer pour basculer ? Lequel appelle ses outils ? Que coûte chacun, en tokens et en temps ? Ces six demandes ne sont pas une évaluation : elles servent à vérifier que la bascule fonctionne. Le jeu figé ne sert qu'à mesurer ; rien n'est réglé sur lui.

In [ ]:
eval_part = eval_fige.groupby(nombre_appels).head(2)
COLONNES = [
  "premier_appel",
  "trouves",
  "appels",
  "generations",
  "secondes",
  "tokens_prompt",
  "tokens_caches",
  "tokens_generes",
]


def compte_tokens(trajet: dict, usage) -> None:
  """Ajoute au trajet les tokens d'une requête : le prompt, dont ceux du cache, et
  les tokens générés."""
  trajet["tokens_prompt"] = trajet.get("tokens_prompt", 0) + usage.prompt_tokens
  details = usage.prompt_tokens_details
  caches = details.cached_tokens if details and details.cached_tokens else 0
  trajet["tokens_caches"] = trajet.get("tokens_caches", 0) + caches
  trajet["tokens_generes"] = trajet.get("tokens_generes", 0) + usage.completion_tokens


def trouves(attendus: list[dict], obtenus: list[dict]) -> int:
  """Les appels attendus qu'on retrouve parmi les obtenus (meme_appel)."""
  restants = list(obtenus)
  n = 0
  for attendu in attendus:
    for obtenu in restants:
      if meme_appel(obtenu, attendu):
        restants.remove(obtenu)
        n += 1
        break
  return n


def mesure(moteur: str, demande: dict, trajet: dict, secondes: float) -> dict:
  """La ligne d'une demande : la première décision juste (le premier appel attendu,
  ou aucun appel), la part des appels attendus trouvés, les appels et générations,
  la durée, les tokens."""
  attendus, appels = demande["appels_attendus"], trajet["appels"]
  premier = (
    meme_appel(appels[0], attendus[0])
    if attendus and appels
    else not attendus and not appels
  )
  return dict(
    moteur=moteur,
    id=demande["id"],
    attendus=[a["name"] for a in attendus],
    obtenus=[a["name"] for a in appels],
    premier_appel=premier,
    trouves=trouves(attendus, appels) / len(attendus)
    if attendus
    else float(not appels),
    appels=len(appels),
    generations=trajet["generations"],
    secondes=secondes,
    tokens_prompt=trajet.get("tokens_prompt", 0),
    tokens_caches=trajet.get("tokens_caches", 0),
    tokens_generes=trajet.get("tokens_generes", 0),
    appels_faits=appels,
  )

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `reponse = moteur.client.chat.completions.create(model=moteur.modele, messages=messages, tools=OUTILS, temperature=0, max_tokens=max_tokens, extra_body=SANS_PENSEE)`, puis `compte_tokens(trajet, reponse.usage)` et `message = reponse.choices[0].message`
- `message.model_dump(exclude_none=True)` : le message, en dictionnaire, avec ses `tool_calls` ; c'est ce que la requête suivante doit renvoyer au serveur
- `appels = [lit_appel(a) for a in message.tool_calls]` ; `isinstance(a["arguments"], dict)` est faux pour des arguments illisibles
- Un résultat : `dict(role="tool", tool_call_id=appel_outil.id, content=json.dumps(observation, ensure_ascii=False))`
- `mesure(nom, demande, trajet, secondes)` : la ligne d'une demande ; gardez-y aussi la conversation (`messages`), l'exercice 4 la relira ; `bascule.groupby("moteur")[COLONNES].mean()` : la moyenne par moteur
- Sans GPU, sur deux processeurs, une demande prend de quelques secondes à une minute avec Qwen3-0.6B ; sur GPU, quelques secondes avec Qwen3-1.7B

```python
async def agent(moteur, mcp, messages, max_tokens=200, bavard=False) -> dict:
  trajet = dict(appels=[], format_lisible=True, generations=0, reponse=REPLI)
  for _ in range(MAX_ETAPES):
    reponse = moteur.client.chat.completions.create(...)
    trajet["generations"] += 1
    compte_tokens(trajet, reponse.usage)
    message = reponse.choices[0].message
    if not message.tool_calls:
      trajet["reponse"] = nettoie(message.content or "") or REPLI
      break
    messages.append(message.model_dump(exclude_none=True))
    appels = [lit_appel(a) for a in message.tool_calls]
    if not all(isinstance(a["arguments"], dict) for a in appels):
      trajet["format_lisible"] = False
      break
    for appel_outil, appel in zip(message.tool_calls, appels):
      observation = await execute(mcp, appel)
      trajet["appels"].append(appel)
      messages.append(dict(role="tool", tool_call_id=..., content=...))
      if bavard:
        print("Action :", appel["name"], appel["arguments"])
        print("Observation :", observation)
  messages.append(dict(role="assistant", content=trajet["reponse"]))
  if bavard:
    print("Réponse :", trajet["reponse"])
  return trajet


lignes = []
async with Client(SERVEUR) as mcp:
  for nom, moteur in [("base", BASE), ("specialise", SPECIALISE)]:
    for demande in eval_part.to_dict("records"):
      debut = time.time()
      messages = conversation(demande["message"])
      trajet = await agent(...)
      ligne = mesure(nom, demande, trajet, time.time() - debut)
      lignes.append(ligne | dict(messages=messages))  # pour l'exercice 4
bascule = pd.DataFrame(lignes)
print(bascule[["moteur", "id", "attendus", "obtenus"]])
bascule.groupby("moteur")[COLONNES].mean().round(2)
```

### Solution

In [ ]:
async def agent(moteur, mcp, messages, max_tokens=200, bavard=False) -> dict:
  """La boucle ReAct du chapitre 2, par l'API : le moteur génère, le serveur MCP
  exécute les appels, jusqu'à une réponse sans appel ou MAX_ETAPES générations.
  `messages` s'allonge des appels, de leurs résultats et de la réponse."""
  trajet = dict(appels=[], format_lisible=True, generations=0, reponse=REPLI)
  for _ in range(MAX_ETAPES):
    reponse = moteur.client.chat.completions.create(
      model=moteur.modele,
      messages=messages,
      tools=OUTILS,
      temperature=0,
      max_tokens=max_tokens,
      extra_body=SANS_PENSEE,
    )
    trajet["generations"] += 1
    compte_tokens(trajet, reponse.usage)
    message = reponse.choices[0].message
    if not message.tool_calls:  # la réponse
      trajet["reponse"] = nettoie(message.content or "") or REPLI
      break
    messages.append(message.model_dump(exclude_none=True))
    appels = [lit_appel(a) for a in message.tool_calls]
    if not all(isinstance(a["arguments"], dict) for a in appels):  # illisible
      trajet["format_lisible"] = False
      break
    for appel_outil, appel in zip(message.tool_calls, appels):
      observation = await execute(mcp, appel)
      trajet["appels"].append(appel)
      messages.append(
        dict(
          role="tool",
          tool_call_id=appel_outil.id,
          content=json.dumps(observation, ensure_ascii=False),
        )
      )
      if bavard:
        print("Action :", appel["name"], appel["arguments"])
        print("Observation :", observation)
  messages.append(dict(role="assistant", content=trajet["reponse"]))
  if bavard:
    print("Réponse :", trajet["reponse"])
  return trajet


lignes = []
async with Client(SERVEUR) as mcp:
  for nom, moteur in [("base", BASE), ("specialise", SPECIALISE)]:
    for demande in eval_part.to_dict("records"):
      debut = time.time()
      messages = conversation(demande["message"])
      trajet = await agent(moteur, mcp, messages)
      ligne = mesure(nom, demande, trajet, time.time() - debut)
      lignes.append(ligne | dict(messages=messages))  # pour l'exercice 4
bascule = pd.DataFrame(lignes)
print(bascule[["moteur", "id", "attendus", "obtenus"]].to_string())
bascule.groupby("moteur")[COLONNES].mean().round(2)

# Rien n'a changé dans l'agent : seul le moteur, une adresse et un nom de modèle. Un
# modèle qui n'appelle pas ses outils répond vite, et invente ce qu'il n'a pas lu ; un
# appel de plus coûte une génération et un résultat de plus dans le prompt. Le cache
# du slot fait l'essentiel du prompt : la consigne et les outils.

## 4. Robustesse

Un modèle, même affiné, écrit parfois un appel faux : un outil qui n'existe pas, une valeur hors de la liste, un numéro de commande qu'il invente. Un outil échoue, ou ne répond pas. L'agent doit le prévoir.

### 4.1 Valider chaque appel avant de l'exécuter

Le serveur MCP valide les arguments de ses outils, mais il laisse passer un argument inconnu (chapitre 2), et ses refus arrivent en anglais, après l'appel. L'agent valide lui-même chaque appel, avant de l'exécuter. Il a déjà `valide(appel)`, écrite à l'exercice 7 du chapitre 2 et reprise telle quelle ci-dessous : les problèmes d'un appel face aux schémas que déclare le serveur (`SCHEMAS`), un outil inconnu ou les erreurs de `jsonschema`. Il lui manque le schéma strict, les arguments illisibles et les règles métier. Écrivez :

- `strict(schema)`, une copie du schéma qui refuse les arguments qu'il ne déclare pas (`"additionalProperties": False`) ; rangez les schémas stricts des cinq outils dans `SCHEMAS_STRICTS` ;
- `verifie(appel, messages)`, la liste des problèmes d'un appel (`{"name", "arguments"}`, comme ceux de `lit_appel`) : des arguments qui ne sont pas un objet JSON, avant tout ; ceux de `valide` ; ceux que seul le schéma strict trouve, les arguments qu'il ne déclare pas ; et une règle métier, qu'aucun schéma ne dit : un numéro de commande (l'argument `numero`) est de la forme `CMD-` et 6 chiffres, et il apparaît dans un message du client ou dans un résultat d'outil réussi (sans clé `erreur`) de `messages` : jamais inventé. Une liste vide pour un appel valide.

Essayez `verifie` sur les appels d'`essais` (ci-dessous), avec la conversation de leur message, à côté de `valide` : que laissait passer `valide` ?

In [ ]:
# Le chapitre 2, exercice 7, tel quel.
def valide(appel: dict) -> list[str]:
  """Les problèmes d'un appel face au schéma du serveur ; vide s'il est valide."""
  if appel["name"] not in SCHEMAS:
    return [f"outil inconnu : {appel['name']}"]
  validateur = jsonschema.Draft202012Validator(SCHEMAS[appel["name"]])
  return [erreur.message for erreur in validateur.iter_errors(appel["arguments"])]


MESSAGE_ESSAIS = (
  "Bonjour, où en est ma commande CMD-103338 ? Et avez-vous le Biclou Urbain 3 en "
  "taille M à Nantes ? Paul Girard"
)
essais = [
  dict(name="chercher_commande", arguments=dict(numero="CMD-103338")),
  dict(name="chercher_commande", arguments=dict(numero="CMD-123456")),
  dict(
    name="verifier_stock",
    arguments=dict(modele="Biclou Urbain 3", taille="XXL", magasin="Nantes"),
  ),
  dict(
    name="verifier_stock",
    arguments=dict(
      modele="Biclou Urbain 3", taille="M", magasin="Nantes", couleur="bleu"
    ),
  ),
  dict(name="annuler_commande", arguments=dict(numero="CMD-103338")),
  dict(name="chercher_commande", arguments='{"numero": "CMD-10'),
]

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `schema | {"additionalProperties": False}` : un nouveau dictionnaire, le schéma et cette clé
- `valide(appel)` : les problèmes que le chapitre 2 relevait déjà, une liste, que `+=` complète
- `jsonschema.Draft202012Validator(schema).iter_errors(arguments)` : les erreurs, chacune avec son `.message` et `.validator`, le mot-clé du schéma qu'elle enfreint : `"additionalProperties"` pour un argument que le schéma ne déclare pas ; les autres, `valide` les a déjà
- Les textes que l'agent a lus : `m["content"]` pour les messages `m` du client (`m["role"] == "user"`) et pour les résultats réussis (`m["role"] == "tool"` et `'"erreur"' not in m["content"]`) ; un refus répète le numéro refusé, il ne compte pas
- `re.fullmatch(r"CMD-\d{6}", numero)` : `None` pour un numéro mal formé
- Exemples à vérifier : le premier essai n'a aucun problème ; le deuxième, un numéro absent de la conversation ; le quatrième, un argument `couleur` refusé ; `valide` ne trouve ni l'un ni l'autre

```python
def strict(schema: dict) -> dict:
  return schema | ...


SCHEMAS_STRICTS = {nom: strict(schema) for nom, schema in SCHEMAS.items()}


def verifie(appel: dict, messages: list[dict]) -> list[str]:
  arguments = appel["arguments"]
  if not isinstance(arguments, dict):
    return [...]
  problemes = valide(appel)
  if appel["name"] not in SCHEMAS_STRICTS:
    return problemes
  validateur = jsonschema.Draft202012Validator(SCHEMAS_STRICTS[appel["name"]])
  problemes += [
    erreur.message
    for erreur in validateur.iter_errors(arguments)
    if erreur.validator == ...
  ]
  lu = " ".join(m["content"] for m in messages if ...)
  numero = arguments.get("numero")
  if isinstance(numero, str):
    if not re.fullmatch(r"CMD-\d{6}", numero):
      problemes.append(f"numéro de commande mal formé : {numero}")
    elif numero not in lu:
      problemes.append(f"numéro de commande absent de la conversation : {numero}")
  return problemes


for appel in essais:
  print(appel["name"], appel["arguments"])
  print("  valide  ->", valide(appel))
  print("  verifie ->", verifie(appel, ...))
```

#### Solution

In [ ]:
def strict(schema: dict) -> dict:
  """Le schéma, qui refuse en plus les arguments qu'il ne déclare pas."""
  return schema | {"additionalProperties": False}


SCHEMAS_STRICTS = {nom: strict(schema) for nom, schema in SCHEMAS.items()}


def verifie(appel: dict, messages: list[dict]) -> list[str]:
  """Les problèmes d'un appel, avant de l'exécuter ; vide s'il est valide."""
  arguments = appel["arguments"]
  if not isinstance(arguments, dict):
    return [f"arguments illisibles, un objet JSON attendu : {arguments}"]
  problemes = valide(appel)  # un outil inconnu, les erreurs face au schéma du serveur
  if appel["name"] not in SCHEMAS_STRICTS:
    return problemes
  # Ce que seul le schéma strict trouve : les arguments qu'il ne déclare pas.
  validateur = jsonschema.Draft202012Validator(SCHEMAS_STRICTS[appel["name"]])
  problemes += [
    erreur.message
    for erreur in validateur.iter_errors(arguments)
    if erreur.validator == "additionalProperties"
  ]
  # La règle métier : un numéro de commande vient du client ou d'un outil qui a réussi.
  lu = " ".join(
    m["content"]
    for m in messages
    if m["role"] == "user" or (m["role"] == "tool" and '"erreur"' not in m["content"])
  )
  numero = arguments.get("numero")
  if isinstance(numero, str):
    if not re.fullmatch(r"CMD-\d{6}", numero):
      problemes.append(f"numéro de commande mal formé : {numero}")
    elif numero not in lu:
      problemes.append(f"numéro de commande absent de la conversation : {numero}")
  return problemes


for appel in essais:
  print(appel["name"], appel["arguments"])
  print("  valide  ->", valide(appel))
  print("  verifie ->", verifie(appel, conversation(MESSAGE_ESSAIS)))

# Le schéma dit la forme d'un appel ; les règles métier, ce qu'un appel a le droit de
# dire : un numéro que personne n'a écrit est inventé, même bien formé. `valide`
# laissait passer l'argument inconnu et le numéro inventé.

### 4.2 La boucle robuste

Écrivez `agent_robuste(moteur, mcp, messages)`, la boucle de l'exercice 3, avec ces règles :

| Panne | Réponse de l'agent |
| --- | --- |
| Appel invalide (`verifie`) | pas exécuté ; l'erreur revient au modèle, en message `tool` : `{"erreur": "appel refusé : ..."}` ; c'est une relance |
| Erreur de l'outil (un résultat qui a une clé `erreur`) | le résultat revient au modèle tel quel ; c'est une relance |
| Outil qui ne répond pas (`execute` lève une exception) | une lecture (`IDEMPOTENTS`) se relance une fois ; une écriture (`creer_ticket`, `escalader`), jamais : on ne sait pas si elle a eu lieu, transfert |
| Écriture déjà faite dans la conversation (le même outil) | pas refaite : le modèle reçoit le premier résultat |
| Serveur du modèle injoignable (`openai.APIError`, après les relances du client) | transfert |
| Plus de `RELANCES_MAX` relances, ou `MAX_ETAPES` générations | transfert |

Un transfert arrête la boucle sur la réponse de repli, `REPLI`, et le trajet en garde la raison (`transfert`, `None` sans transfert) : un conseiller reprend la demande. La fonction rend le trajet de l'exercice 3, avec en plus ses `refus` (les problèmes des appels refusés), ses `relances` et son `transfert`.

Côté modèle, le client `openai` relance déjà de lui-même une requête qui échoue (connexion, délai dépassé, erreurs 429 et 5xx du serveur), deux fois par défaut, avec une attente qui croît (`max_retries`), et coupe une requête après `timeout` secondes : sans risque, une génération n'écrit rien. Côté outils, c'est l'agent qui décide : une lecture peut se relancer, une écriture relancée à l'aveugle crée deux tickets.

**Documentation utile**

- [Les relances et les délais du client `openai`](https://github.com/openai/openai-python#retries)

In [ ]:
IDEMPOTENTS = {"chercher_commande", "verifier_stock", "chercher_politique"}
RELANCES_MAX = 2  # erreurs renvoyées au modèle au plus, par message du client


def refus(problemes: list[str]) -> dict:
  return {"erreur": "appel refusé : " + " ; ".join(problemes)}

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `try: ... except openai.APIError:` : toutes les erreurs du client `openai`, connexion et délai compris
- `faits = {}` : les écritures déjà faites, par nom d'outil, avec leur résultat ; `appel["name"] in faits` avant d'exécuter
- Une lecture relancée : `for essai in range(2): try: observation = await execute(mcp, appel); break except Exception: ...`
- Le message `tool` d'un appel refusé ou d'un résultat porte le `tool_call_id` de l'appel, comme dans l'exercice 3
- Les relances se comptent sur tout le message du client, pas par appel

```python
async def agent_robuste(moteur, mcp, messages, max_tokens=200, bavard=False) -> dict:
  trajet = dict(
    appels=[],
    format_lisible=True,
    generations=0,
    reponse=REPLI,
    refus=[],
    relances=0,
    transfert=None,
  )
  faits = {}  # les écritures exécutées, et leur résultat
  for _ in range(MAX_ETAPES):
    try:
      reponse = moteur.client.chat.completions.create(...)
    except openai.APIError as erreur:
      trajet["transfert"] = f"serveur du modèle : {type(erreur).__name__}"
      break
    trajet["generations"] += 1
    compte_tokens(trajet, reponse.usage)
    message = reponse.choices[0].message
    if not message.tool_calls:
      trajet["reponse"] = nettoie(message.content or "") or REPLI
      break
    messages.append(message.model_dump(exclude_none=True))
    for appel_outil in message.tool_calls:
      appel = lit_appel(appel_outil)
      problemes = verifie(appel, messages)
      if problemes:
        observation = refus(problemes)
        trajet["refus"] += problemes
      elif appel["name"] in faits:
        observation = ...
      else:
        observation = None
        for essai in range(2 if appel["name"] in IDEMPOTENTS else 1):
          try:
            observation = await execute(mcp, appel)
            break
          except Exception:
            pass
        if observation is None:
          trajet["transfert"] = f"{appel['name']} sans réponse"
          break
        trajet["appels"].append(appel)
        if appel["name"] not in IDEMPOTENTS:
          faits[appel["name"]] = observation
      if "erreur" in observation:
        trajet["relances"] += 1
      messages.append(dict(role="tool", tool_call_id=..., content=...))
      if bavard:
        print("Action :", appel["name"], appel["arguments"], "->", observation)
    if trajet["transfert"] is None and trajet["relances"] > RELANCES_MAX:
      trajet["transfert"] = "trop de relances"
    if trajet["transfert"]:
      break
  else:
    trajet["transfert"] = "budget d'étapes épuisé"
  if trajet["transfert"]:
    trajet["reponse"] = REPLI
  messages.append(dict(role="assistant", content=trajet["reponse"]))
  if bavard:
    print("Transfert :", trajet["transfert"], "| Réponse :", trajet["reponse"][:80])
  return trajet
```

#### Solution

In [ ]:
async def agent_robuste(moteur, mcp, messages, max_tokens=200, bavard=False) -> dict:
  """La boucle de l'exercice 3, qui valide chaque appel avant de l'exécuter, renvoie
  au modèle les refus et les erreurs des outils, ne refait jamais une écriture, et
  passe la main à un conseiller quand elle ne peut pas conclure."""
  trajet = dict(
    appels=[],
    format_lisible=True,
    generations=0,
    reponse=REPLI,
    refus=[],
    relances=0,
    transfert=None,
  )
  faits = {}  # les écritures exécutées, et leur résultat
  for _ in range(MAX_ETAPES):
    try:
      reponse = moteur.client.chat.completions.create(
        model=moteur.modele,
        messages=messages,
        tools=OUTILS,
        temperature=0,
        max_tokens=max_tokens,
        extra_body=SANS_PENSEE,
      )
    except openai.APIError as erreur:  # après les relances du client
      trajet["transfert"] = f"serveur du modèle : {type(erreur).__name__}"
      break
    trajet["generations"] += 1
    compte_tokens(trajet, reponse.usage)
    message = reponse.choices[0].message
    if not message.tool_calls:  # la réponse
      trajet["reponse"] = nettoie(message.content or "") or REPLI
      break
    messages.append(message.model_dump(exclude_none=True))
    for appel_outil in message.tool_calls:
      appel = lit_appel(appel_outil)
      problemes = verifie(appel, messages)
      if problemes:  # refusé avant toute exécution
        observation = refus(problemes)
        trajet["refus"] += problemes
      elif appel["name"] in faits:  # une écriture déjà faite : pas deux fois
        observation = faits[appel["name"]]
      else:
        observation = None
        for _ in range(2 if appel["name"] in IDEMPOTENTS else 1):
          try:
            observation = await execute(mcp, appel)
            break
          except Exception:  # l'outil n'a pas répondu
            pass
        if observation is None:  # une écriture a peut-être eu lieu
          trajet["transfert"] = f"{appel['name']} sans réponse"
          break
        trajet["appels"].append(appel)
        if appel["name"] not in IDEMPOTENTS:
          faits[appel["name"]] = observation
      if "erreur" in observation:
        trajet["relances"] += 1
      messages.append(
        dict(
          role="tool",
          tool_call_id=appel_outil.id,
          content=json.dumps(observation, ensure_ascii=False),
        )
      )
      if bavard:
        print("Action :", appel["name"], appel["arguments"], "->", observation)
    if trajet["transfert"] is None and trajet["relances"] > RELANCES_MAX:
      trajet["transfert"] = "trop de relances"
    if trajet["transfert"]:
      break
  else:
    trajet["transfert"] = "budget d'étapes épuisé"
  if trajet["transfert"]:
    trajet["reponse"] = REPLI
  messages.append(dict(role="assistant", content=trajet["reponse"]))
  if bavard:
    print("Transfert :", trajet["transfert"], "| Réponse :", trajet["reponse"][:80])
  return trajet

### 4.3 Pannes simulées, puis le vrai modèle

Une panne se provoque pour vérifier qu'on la gère. Deux doublures (cellule ci-dessous) : `DoublureLLM(script)`, un faux client `openai` qui rend, dans l'ordre, les sorties de son script (`sortie(...)`, des appels ou un texte) ; `MCPEnPanne(mcp, perdus)`, le client MCP dont les outils de `perdus` s'exécutent, puis ne répondent pas, et qui compte les exécutions réelles (`executions`). `SCENARIOS` : sept pannes, chacune un message, un script et les outils perdus. Aucune n'exerce la deuxième ligne du tableau, l'erreur de l'outil : ajoutez la huitième, `"erreur de l'outil"`, un client qui cite une commande qui n'existe pas, `CMD-999999` (`chercher_commande` rend alors `{"erreur": ...}`), et un modèle qui la cherche, puis répond. Attendu : une relance, aucun transfert, une seule exécution.

Faites tourner `agent_robuste` sur chaque scénario, la doublure en guise de moteur (`Moteur(DoublureLLM(script), "doublure")`) et le client MCP enveloppé dans `MCPEnPanne`, avec la trace (`bavard=True`), et affichez les relances, le transfert et les outils réellement exécutés. Puis le serveur du modèle injoignable : `ABSENT` (ci-dessous) vise un port où rien n'écoute.

Enfin, le vrai modèle : faites tourner `agent_robuste` avec `SPECIALISE` sur `eval_part`, rangez ses lignes (`mesure`, avec en plus ses `refus`, `relances` et `transfert`) dans `robuste`, et comparez-les à celles du moteur spécialisé de l'exercice 3. Combien des appels réels de l'exercice 3 `verifie` aurait-elle refusés, pour chaque moteur ? Que coûte la robustesse, en générations et en temps ?

In [ ]:
def sortie(*appels: tuple[str, str], texte: str = "") -> dict:
  """Une sortie de la doublure : des appels (nom, arguments en texte), ou un texte."""
  appels_outils = [
    dict(id=f"appel-{i}", type="function", function=dict(name=nom, arguments=args))
    for i, (nom, args) in enumerate(appels)
  ]
  return dict(content=texte, tool_calls=appels_outils or None)


class DoublureLLM:
  """Un faux client openai qui rend, dans l'ordre, les sorties de son script."""

  def __init__(self, script: list[dict]):
    self.script = list(script)
    self.chat = self.completions = self  # doublure.chat.completions.create

  def create(self, **requete) -> ChatCompletion:
    message = dict(role="assistant", **self.script.pop(0))
    return ChatCompletion.model_validate(
      dict(
        id="doublure",
        object="chat.completion",
        created=0,
        model="doublure",
        choices=[dict(index=0, finish_reason="stop", message=message)],
        usage=dict(prompt_tokens=0, completion_tokens=0, total_tokens=0),
      )
    )


class MCPEnPanne:
  """Le client MCP, dont les outils `perdus` s'exécutent, puis ne répondent pas."""

  def __init__(self, mcp, perdus=()):
    self.mcp, self.perdus, self.executions = mcp, set(perdus), []

  async def call_tool(self, nom: str, arguments: dict):
    resultat = await self.mcp.call_tool(nom, arguments)
    self.executions.append(nom)
    if nom in self.perdus:
      raise TimeoutError(f"{nom} : pas de réponse du serveur d'outils")
    return resultat


COMMANDE = '{"numero": "CMD-103338"}'
TICKET = '{"categorie": "livraison", "priorite": "normale", "resume": "colis abîmé"}'
REPONSE = sortie(texte="Bonjour,\n\n(réponse au client)\n\nBonne journée,")
SCENARIOS = {
  "outil inconnu, puis corrigé": (
    "Ma commande CMD-103338 est-elle partie ?",
    [
      sortie(("annuler_commande", COMMANDE)),
      sortie(("chercher_commande", COMMANDE)),
      REPONSE,
    ],
    [],
  ),
  "valeur hors liste et argument inconnu": (
    "Avez-vous le Biclou Urbain 3 en M à Nantes ?",
    [
      sortie(
        (
          "verifier_stock",
          '{"modele": "Biclou Urbain 3", "taille": "Medium", "magasin": "Nantes", "couleur": "bleu"}',
        )
      ),
      sortie(
        (
          "verifier_stock",
          '{"modele": "Biclou Urbain 3", "taille": "M", "magasin": "Nantes"}',
        )
      ),
      REPONSE,
    ],
    [],
  ),
  "numéro inventé, trois fois": (
    "Où en est ma commande ?",
    [sortie(("chercher_commande", '{"numero": "CMD-123456"}'))] * 3 + [REPONSE],
    [],
  ),
  "arguments illisibles, puis lus": (
    "Ma commande CMD-103338 est-elle partie ?",
    [
      sortie(("chercher_commande", '{"numero": "CMD-10')),
      sortie(("chercher_commande", COMMANDE)),
      REPONSE,
    ],
    [],
  ),
  "ticket demandé deux fois": (
    "Mon colis CMD-103338 est arrivé abîmé.",
    [sortie(("creer_ticket", TICKET)), sortie(("creer_ticket", TICKET)), REPONSE],
    [],
  ),
  "lecture sans réponse": (
    "Ma commande CMD-103338 est-elle partie ?",
    [sortie(("chercher_commande", COMMANDE)), REPONSE],
    ["chercher_commande"],
  ),
  "ticket créé, réponse perdue": (
    "Mon colis CMD-103338 est arrivé abîmé.",
    [sortie(("creer_ticket", TICKET)), REPONSE],
    ["creer_ticket"],
  ),
}
# Un serveur de modèle injoignable : une relance du client, des délais courts.
ABSENT = Moteur(
  OpenAI(
    base_url="http://localhost:8099/v1", api_key="aucune", max_retries=1, timeout=5
  ),
  "absent",
)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- Un scénario : `message, script, perdus = SCENARIOS[nom]`, puis `panne = MCPEnPanne(mcp, perdus)` et `await agent_robuste(Moteur(DoublureLLM(script), "doublure"), panne, conversation(message), bavard=True)` ; `panne.executions` : les outils réellement exécutés
- Le huitième scénario, comme les autres, un triplet `(message, script, perdus)` : un script de deux sorties, l'appel de `chercher_commande` sur `CMD-999999`, puis `REPONSE` ; aucun outil perdu
- `ABSENT` n'a pas besoin du serveur MCP : `await agent_robuste(ABSENT, mcp, conversation(...))`
- Une ligne de `robuste` : `mesure("robuste", demande, trajet, secondes) | dict(refus=len(trajet["refus"]), relances=trajet["relances"], transfert=trajet["transfert"])`
- Les appels réels de l'exercice 3 : les colonnes `appels_faits` et `messages` de `bascule` ; `verifie(appel, messages)` valide un appel avec la conversation de sa demande

```python
SCENARIOS["erreur de l'outil"] = (..., [sortie(...), REPONSE], [])
async with Client(SERVEUR) as mcp:
  for nom, (message, script, perdus) in SCENARIOS.items():
    print(f"\n=== {nom}")
    panne = MCPEnPanne(mcp, perdus)
    trajet = await agent_robuste(...)
    print("Relances :", trajet["relances"], "| exécutés :", panne.executions)
  print("\n=== Serveur du modèle injoignable")
  trajet = await agent_robuste(ABSENT, mcp, conversation("Bonjour"))
  print("Transfert :", trajet["transfert"])

  lignes_robustes = []
  for demande in eval_part.to_dict("records"):
    debut = time.time()
    trajet = await agent_robuste(SPECIALISE, mcp, conversation(demande["message"]))
    ligne = mesure("robuste", demande, trajet, time.time() - debut)
    lignes_robustes.append(ligne | dict(...))
robuste = pd.DataFrame(lignes_robustes)
print(robuste[["id", "attendus", "obtenus", "refus", "relances", "transfert"]])
...
```

#### Solution

In [ ]:
# Le huitième scénario : une commande que le client cite, mais qui n'existe pas.
SCENARIOS["erreur de l'outil"] = (
  "Bonjour, où en est ma commande CMD-999999 ?",
  [sortie(("chercher_commande", '{"numero": "CMD-999999"}')), REPONSE],
  [],
)
async with Client(SERVEUR) as mcp:
  for nom, (message, script, perdus) in SCENARIOS.items():
    print(f"\n=== {nom}")
    panne = MCPEnPanne(mcp, perdus)
    doublure = Moteur(DoublureLLM(script), "doublure")
    trajet = await agent_robuste(doublure, panne, conversation(message), bavard=True)
    print("Relances :", trajet["relances"], "| exécutés :", panne.executions)
  print("\n=== Serveur du modèle injoignable")
  trajet = await agent_robuste(ABSENT, mcp, conversation(MESSAGE_ESSAIS))
  print("Transfert :", trajet["transfert"])

  lignes_robustes = []
  for demande in eval_part.to_dict("records"):
    debut = time.time()
    trajet = await agent_robuste(SPECIALISE, mcp, conversation(demande["message"]))
    ligne = mesure("robuste", demande, trajet, time.time() - debut)
    ligne |= dict(
      refus=len(trajet["refus"]),
      relances=trajet["relances"],
      transfert=trajet["transfert"],
    )
    lignes_robustes.append(ligne)
robuste = pd.DataFrame(lignes_robustes)
print()
print(
  robuste[["id", "attendus", "obtenus", "refus", "relances", "transfert"]].to_string()
)

# Les appels réels de l'exercice 3, validés après coup, avec leur conversation.
refuses = {"base": [], "specialise": []}
for ligne in lignes:
  for appel in ligne["appels_faits"]:
    refuses[ligne["moteur"]] += verifie(appel, ligne["messages"])
for moteur, problemes in refuses.items():
  print(f"\nExercice 3, {moteur} : {len(problemes)} problèmes", problemes[:3])
comparaison = pd.concat([bascule[bascule["moteur"] == "specialise"], robuste])
comparaison.groupby("moteur")[COLONNES + ["refus", "relances"]].mean().round(2)

# Chaque panne a sa réponse : refusée, l'erreur revient au modèle, qui corrige ; trop
# d'erreurs, ou une écriture sans réponse, et un conseiller reprend la main. Le ticket
# perdu n'a été créé qu'une fois. L'erreur de l'outil revient au modèle, qui répond :
# une relance, une exécution, pas de transfert. La robustesse coûte une génération
# par relance ; compter les refus, les relances et les transferts signale un modèle
# qui dérive.

## 5. Architecture mixte

Le modèle spécialisé traite bien ce qu'il a appris : un message, un besoin, ses appels. Une demande composée (une commande en retard et un vélo à réserver, dans le même message) le sort de ce qu'il a vu. L'architecture mixte confie cette part à un modèle généraliste, plus grand :

1. un **routeur** envoie droit à l'agent spécialisé les demandes à un seul besoin, la plus grande part du trafic ;
2. le **généraliste** découpe une demande composée en sous-demandes à un seul besoin, par une sortie structurée ;
3. l'**agent spécialisé** traite chaque sous-demande, ses appels d'outils, comme à l'entraînement ;
4. le **généraliste** rédige la réponse au client à partir des résultats.

Le généraliste (cellule ci-dessous) : un modèle par API si vous en avez un, la clé dans le secret Colab `OPENAI_API_KEY` (et, en option, `OPENAI_BASE_URL` pour un autre fournisseur compatible, `GENERALISTE_MODELE` pour le nom du modèle) ; sinon Qwen3-4B-Instruct-2507, quantifié en Q4_K_M, servi ici même (2,5 Go à télécharger, lent sur deux processeurs). Le routeur, `composee`, est une règle : elle reconnaît les demandes composées de ce TP, construites comme au chapitre 2 (« Autre chose : »), et en raterait de vraies ; un routeur s'apprend sur l'historique, ou se fait par un appel court à un petit modèle.

Écrivez :

- `generaliste(messages, trajet, **options)`, le texte de la réponse du généraliste à `messages` (avec `OPTIONS_GENERALISTE` et les `options`), qui ajoute ses tokens, prompt et génération, à `trajet["tokens_generaliste"]` ;
- `planifie(message, trajet)`, le plan du généraliste pour un message : `CONSIGNES_PLAN` en message `system`, une sortie structurée au schéma de `Plan` (`FORMAT_PLAN`), validée par `Plan` ; `None` si elle n'est pas un plan valide ;
- `mixte(mcp, message)`, une fonction `async` : une demande simple (`composee` faux) va à `agent_robuste` avec `SPECIALISE`, qui y répond ; une demande composée est planifiée, chaque sous-demande traitée par `agent_robuste` avec `SPECIALISE` dans sa propre conversation (96 tokens au plus : sa réponse n'est pas lue), puis le généraliste rédige la réponse (`CONSIGNES_REDACTION`) à partir du message et, pour chaque sous-demande, des résultats de ses outils ; un plan invalide ou vide, un transfert dans une sous-demande ou le généraliste injoignable (`openai.APIError`), et la demande entière va à un conseiller (`REPLI`). Elle rend un trajet : sa `route` (`"specialise"` ou `"mixte"`), ses `appels`, ses `tokens_generaliste`, son `transfert` et sa `reponse`.

Comparez sur `cas_mixtes` (ci-dessous : deux demandes composées de l'historique, et deux demandes simples du jeu figé) : l'agent spécialisé seul (`agent_robuste`) sur les demandes composées, puis `mixte` sur toutes, une ligne par demande et par approche, avec la part des appels attendus trouvés (`trouves`), les appels, les tokens du généraliste (le coût d'un modèle par API) et la durée. Qu'apporte le généraliste sur une demande composée, à quel prix ? Que coûte une demande simple ?

**Documentation utile**

- [`ConfigDict.extra` de Pydantic](https://docs.pydantic.dev/latest/api/config/#pydantic.config.ConfigDict.extra)

In [ ]:
arrete("qwen3-base")  # le modèle de base a fini son rôle : sa mémoire va au généraliste
cle = secret("OPENAI_API_KEY")
if cle:
  client = OpenAI(api_key=cle, base_url=secret("OPENAI_BASE_URL"), timeout=120)
  GENERALISTE = Moteur(client, secret("GENERALISTE_MODELE") or "gpt-5.4-mini")
  OPTIONS_GENERALISTE = {}
else:
  chemin = hf_hub_download(
    "unsloth/Qwen3-4B-Instruct-2507-GGUF",
    "Qwen3-4B-Instruct-2507-Q4_K_M.gguf",
    revision="a06e946bb6b655725eafa393f4a9745d460374c9",
  )
  demarre_serveur("generaliste", 8082, gguf=chemin)
  client = OpenAI(base_url="http://localhost:8082/v1", api_key="aucune")
  GENERALISTE = Moteur(client, "generaliste")
  OPTIONS_GENERALISTE = dict(temperature=0, max_tokens=300)
print("Généraliste :", GENERALISTE.modele)


class Plan(BaseModel):
  """Le plan du généraliste : des sous-demandes à un seul besoin chacune."""

  model_config = ConfigDict(extra="forbid")  # "additionalProperties": false
  sous_demandes: list[str]


FORMAT_PLAN = {
  "type": "json_schema",
  "json_schema": {"name": "plan", "schema": Plan.model_json_schema(), "strict": True},
}
CONSIGNES_PLAN = """Tu prépares le travail de l'agent du service client de Biclou & Cie, qui vend, répare et loue des vélos. L'agent traite très bien les messages qui n'ont qu'un besoin. Découpe le message du client en messages à un seul besoin chacun, comme si le client les avait écrits l'un après l'autre : reprends ses mots, avec, dans chaque message, ce dont il a besoin (numéro de commande, modèle, taille, magasin) et la signature du client. Un message qui n'a qu'un besoin donne un seul message, inchangé."""
# « Ta réponse suit la charte : … », sans sa dernière clause, « et en anglais pour un
# message en anglais » : la rédaction dit déjà « dans la langue du message », et avec
# les deux, le généraliste ajoutait une copie anglaise à une réponse en français.
CLAUSE_ANGLAIS = ", et en anglais pour un message en anglais"
REGLES_REPONSE = CONSIGNES_AGENT.split("\n\n")[-1]
assert CLAUSE_ANGLAIS in REGLES_REPONSE, "la consigne de l'agent a changé : relisez-la"
REGLES_REPONSE = REGLES_REPONSE.replace(CLAUSE_ANGLAIS, "")
CONSIGNES_REDACTION = (
  "Tu rédiges la réponse du service client de Biclou & Cie à un message de client, à "
  "partir des résultats des outils de l'agent, sans rien affirmer qu'ils ne disent : "
  "une seule réponse, dans la langue du message. " + REGLES_REPONSE
)


def composee(message: str) -> bool:
  """Le routeur : vrai pour un message qui ajoute une seconde demande."""
  return bool(re.search(r"\b(autre chose|par ailleurs|et aussi)\b", message, re.I))


# Deux demandes composées, réunies comme au chapitre 2 : deux messages de chat de
# l'historique, à un appel chacun, de scénarios différents.
simples = demandes[
  (demandes["canal"] == "chat")
  & (demandes["appels_attendus"].str.len() == 1)
  & ~demandes["message"].str.contains("\n")
]
cas_mixtes = []
for graine in range(10):
  a, b = simples.sample(2, random_state=graine).to_dict("records")
  if a["scenario"] != b["scenario"] and len(cas_mixtes) < 2:
    second = re.sub(r"^(Bonjour|Bjr|Salut)\W*", "", b["message"])
    message = f"{a['message']} Autre chose : {second}"
    cas_mixtes.append(
      dict(message=message, appels_attendus=a["appels_attendus"] + b["appels_attendus"])
    )
cas_mixtes += eval_part[eval_part["appels_attendus"].str.len() == 1].to_dict("records")
for cas in cas_mixtes:
  print("composée" if composee(cas["message"]) else "simple", ":", cas["message"])

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `GENERALISTE.client.chat.completions.create(model=GENERALISTE.modele, messages=messages, **OPTIONS_GENERALISTE, **options)` ; `.usage.total_tokens` : les tokens du prompt et de la génération
- Le plan : `generaliste(conversation(message, CONSIGNES_PLAN), trajet, response_format=FORMAT_PLAN)`, puis `Plan.model_validate_json(texte)`, qui lève `ValidationError` pour un texte qui n'est pas un plan
- Les résultats d'une sous-demande : `[json.loads(m["content"]) for m in messages if m["role"] == "tool"]`, sur les messages de sa conversation après `agent_robuste`
- La rédaction : `generaliste([dict(role="system", content=CONSIGNES_REDACTION), dict(role="user", content=f"Message du client :\n{message}\n\nRésultats des outils :\n{json.dumps(resultats, ensure_ascii=False, indent=1)}")], trajet)`
- Une ligne : `dict(approche=..., attendus=len(cas["appels_attendus"]), trouves=trouves(cas["appels_attendus"], trajet["appels"]), appels=len(trajet["appels"]), tokens_generaliste=trajet.get("tokens_generaliste", 0), secondes=..., route=...)`
- Sur deux processeurs, le généraliste local écrit quelques tokens par seconde : une demande composée prend quelques minutes ; sur GPU, bien moins

```python
def generaliste(messages: list[dict], trajet: dict, **options) -> str:
  reponse = GENERALISTE.client.chat.completions.create(...)
  trajet["tokens_generaliste"] += reponse.usage.total_tokens
  return reponse.choices[0].message.content or ""


def planifie(message: str, trajet: dict) -> Plan | None:
  texte = generaliste(...)
  try:
    return Plan.model_validate_json(texte)
  except ValidationError:
    return None


async def mixte(mcp, message: str) -> dict:
  trajet = dict(route="specialise", appels=[], tokens_generaliste=0, transfert=None)
  if not composee(message):
    seul = await agent_robuste(SPECIALISE, mcp, conversation(message))
    return trajet | dict(appels=..., transfert=..., reponse=...)
  trajet["route"] = "mixte"
  plan = planifie(message, trajet)  # dans un try : le généraliste peut être injoignable
  if plan is None or not plan.sous_demandes:
    return trajet | dict(transfert="plan invalide", reponse=REPLI)
  resultats = []
  for sous_demande in plan.sous_demandes:
    messages = conversation(sous_demande)
    sous_trajet = await agent_robuste(SPECIALISE, mcp, messages, max_tokens=96)
    trajet["appels"] += sous_trajet["appels"]
    if sous_trajet["transfert"]:
      return trajet | dict(transfert=sous_trajet["transfert"], reponse=REPLI)
    outils = [json.loads(m["content"]) for m in messages if m["role"] == "tool"]
    resultats.append(dict(demande=sous_demande, resultats=outils))
  redaction = [
    dict(role="system", content=CONSIGNES_REDACTION),
    dict(role="user", content=...),
  ]
  return trajet | dict(reponse=nettoie(generaliste(redaction, trajet)) or REPLI)


lignes_mixtes = []
async with Client(SERVEUR) as mcp:
  for cas in cas_mixtes:
    approches = ["specialise seul", "mixte"] if composee(cas["message"]) else ["mixte"]
    for approche in approches:
      debut = time.time()
      if approche == "mixte":
        trajet = await mixte(mcp, cas["message"])
      else:
        trajet = await agent_robuste(...)
      lignes_mixtes.append(dict(approche=approche, trouves=..., secondes=...))
      print(f"\n--- {approche} : {cas['message']}\n{trajet['reponse']}")
pd.DataFrame(lignes_mixtes)
```

### Solution

In [ ]:
def generaliste(messages: list[dict], trajet: dict, **options) -> str:
  """La réponse du généraliste ; ses tokens s'ajoutent au trajet."""
  reponse = GENERALISTE.client.chat.completions.create(
    model=GENERALISTE.modele, messages=messages, **OPTIONS_GENERALISTE, **options
  )
  trajet["tokens_generaliste"] += reponse.usage.total_tokens
  return reponse.choices[0].message.content or ""


def planifie(message: str, trajet: dict) -> Plan | None:
  """Le plan du généraliste, validé, ou None."""
  texte = generaliste(
    conversation(message, CONSIGNES_PLAN), trajet, response_format=FORMAT_PLAN
  )
  try:
    return Plan.model_validate_json(texte)
  except ValidationError:
    return None


async def mixte(mcp, message: str) -> dict:
  """Le routeur, le généraliste qui planifie, le spécialisé qui exécute chaque
  sous-demande, et le généraliste qui répond."""
  trajet = dict(route="specialise", appels=[], tokens_generaliste=0, transfert=None)
  if not composee(message):  # la plus grande part du trafic
    seul = await agent_robuste(SPECIALISE, mcp, conversation(message))
    return trajet | {cle: seul[cle] for cle in ["appels", "transfert", "reponse"]}
  trajet["route"] = "mixte"
  try:
    plan = planifie(message, trajet)
    if plan is None or not plan.sous_demandes:
      return trajet | dict(transfert="plan invalide", reponse=REPLI)
    resultats = []
    for sous_demande in plan.sous_demandes:
      messages = conversation(sous_demande)
      sous_trajet = await agent_robuste(SPECIALISE, mcp, messages, max_tokens=96)
      trajet["appels"] += sous_trajet["appels"]
      if sous_trajet["transfert"]:  # un conseiller reprend tout le message
        return trajet | dict(transfert=sous_trajet["transfert"], reponse=REPLI)
      outils = [json.loads(m["content"]) for m in messages if m["role"] == "tool"]
      resultats.append(dict(demande=sous_demande, resultats=outils))
    contenu = (
      f"Message du client :\n{message}\n\nRésultats des outils :\n"
      + json.dumps(resultats, ensure_ascii=False, indent=1)
    )
    redaction = [
      dict(role="system", content=CONSIGNES_REDACTION),
      dict(role="user", content=contenu),
    ]
    texte = generaliste(redaction, trajet)
  except openai.APIError as erreur:  # le généraliste injoignable : un conseiller
    return trajet | dict(
      transfert=f"généraliste : {type(erreur).__name__}", reponse=REPLI
    )
  return trajet | dict(reponse=nettoie(texte) or REPLI)


lignes_mixtes = []
async with Client(SERVEUR) as mcp:
  for cas in cas_mixtes:
    approches = ["specialise seul", "mixte"] if composee(cas["message"]) else ["mixte"]
    for approche in approches:
      debut = time.time()
      if approche == "mixte":
        trajet = await mixte(mcp, cas["message"])
      else:
        trajet = await agent_robuste(SPECIALISE, mcp, conversation(cas["message"]))
      lignes_mixtes.append(
        dict(
          approche=approche,
          route=trajet.get("route", "specialise"),
          attendus=len(cas["appels_attendus"]),
          trouves=trouves(cas["appels_attendus"], trajet["appels"]),
          appels=len(trajet["appels"]),
          tokens_generaliste=trajet.get("tokens_generaliste", 0),
          transfert=trajet["transfert"],
          secondes=round(time.time() - debut),
        )
      )
      print(f"\n--- {approche} : {cas['message']}\n{trajet['reponse']}")
pd.DataFrame(lignes_mixtes)

# Sur une demande composée, le généraliste coûte deux appels (le plan et la réponse),
# et le spécialisé, une conversation par sous-demande ; une demande simple ne coûte
# rien au généraliste. Le généraliste écrit la réponse sans appeler d'outil : il ne
# peut affirmer que ce que les résultats disent. Deux modèles à servir, à mesurer et
# à faire évoluer : l'architecture se justifie quand les demandes composées sont
# rares et que le spécialisé fait mieux, ou moins cher, sur les simples.

## Arrêter les serveurs

Les serveurs tournent jusqu'à la fin de la session : arrêtez-les quand vous avez fini.

In [ ]:
for nom in list(SERVEURS):
  arrete(nom)
print("Serveurs arrêtés")

## À retenir

- Un serveur d'inférence compatible avec l'API d'OpenAI (llama.cpp, sur GPU ou sur processeur ; vLLM, le serveur des GPU en production) sert le modèle de base et son adaptateur LoRA, applique le gabarit et lit les appels d'outils : l'agent reçoit des `tool_calls`, plus de texte à analyser.
- Servir comme à l'entraînement : le même gabarit (`--chat-template-file`), la même consigne, les mêmes schémas d'outils, dans le même ordre ; on le vérifie en comparant les prompts, sur une demande à plusieurs appels. Ce qu'aucun serveur ne transmet (la clé `return` des outils) se retire du jeu d'entraînement.
- Basculer l'agent, c'est changer de moteur, une adresse et un nom de modèle : la boucle, les outils et la consigne ne changent pas.
- Valider chaque appel avant de l'exécuter, son schéma strict et les règles métier ; renvoyer refus et erreurs au modèle, en nombre borné ; relancer une lecture, jamais une écriture à l'aveugle ; passer la main à un conseiller quand l'agent ne peut pas conclure. Une panne se provoque pour vérifier qu'on la gère.
- L'architecture mixte confie au généraliste ce qui sort de l'apprentissage du spécialisé (découper, rédiger à partir des résultats) et garde le spécialisé pour la tâche fréquente : un routeur, une sortie structurée, deux modèles à mesurer, en qualité, en coût et en latence, au chapitre suivant.